# CV Check - two-source training diagnostic

Evaluates the current `config.json` training layout with video-level `StratifiedGroupKFold`.

Expected source videos: 37 total = 36 wetlandbirds + 1 duck/manual.
Feature B (`mean+std`) is the default. The manual test fold is marked explicitly.


## 1. Settings


In [24]:
FOLDS = 5
INCLUDE_STD = True
RANDOM_STATE = 42
CONFIG_PATH = "config.json"


## 2. Imports and project root


In [25]:
import json
import sys
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import LabelEncoder

ROOT = Path.cwd()
if (ROOT / "src").exists():
    root = ROOT
elif (ROOT.parent / "src").exists():
    root = ROOT.parent
else:
    raise RuntimeError("Run this notebook from the repo root or notebooks/.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from src.classifier import build_feature_matrix
OUTPUT_DIR = root / "notebooks" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("root:", root)


root: C:\Users\osca0\Github\be-more-agent-duck


## 3. Collect the two data sources


In [26]:
def load_config(path):
    config_path = Path(path)
    if not config_path.is_absolute():
        config_path = root / config_path
    with config_path.open("r", encoding="utf-8") as f:
        return json.load(f)

def resolve_path(path_value):
    path = Path(path_value)
    return path if path.is_absolute() else root / path

def collect_source(name, labels_dir, videos_dir):
    labels_dir = resolve_path(labels_dir)
    videos_dir = resolve_path(videos_dir)
    rows = []
    missing_videos = []
    for label_path in sorted(labels_dir.glob("labels_*.csv")):
        video_stem = label_path.stem[len("labels_"):]
        video_path = videos_dir / f"{video_stem}.mp4"
        if video_path.exists():
            rows.append({"source": name, "video_stem": video_stem, "video_path": video_path, "label_path": label_path})
        else:
            missing_videos.append(video_path)
    if missing_videos:
        print(f"[{name}] labels without mp4 skipped:")
        for video_path in missing_videos:
            print("  -", video_path)
    return rows

config = load_config(CONFIG_PATH)
clip_model = config.get("clip_model", "ViT-B/32")
frames_per_seg = int(config.get("frames_per_seg", 10))

sources = []
sources.extend(collect_source("wetlandbirds", config.get("train_labels_dir", "data/train/wetlandbirds"), config.get("train_videos_dir", "data/train/wetlandbirds/videos")))
sources.extend(collect_source("duck", config.get("duck_labels_dir", "data/train/duck"), config.get("duck_labels_dir", "data/train/duck")))
if not sources:
    raise FileNotFoundError("No training label/video pairs found in configured sources.")

source_table = pd.DataFrame(sources)
if source_table["video_stem"].duplicated().any():
    display(source_table[source_table["video_stem"].duplicated(keep=False)])
    raise RuntimeError("Duplicate video_stem values would make source mapping ambiguous.")

video_paths = source_table["video_path"].astype(str).tolist()
labels_csvs = source_table["label_path"].astype(str).tolist()
source_by_group = dict(zip(source_table["video_stem"], source_table["source"]))

box_paths = [resolve_path(p) for p in [config.get("train_boxes_csv"), config.get("duck_boxes_csv")] if p]
missing_boxes = [p for p in box_paths if not p.exists()]
if missing_boxes:
    raise FileNotFoundError(f"Missing box CSV(s): {missing_boxes}")

merged_boxes_csv = OUTPUT_DIR / "cv_check_merged_boxes.csv"
pd.concat([pd.read_csv(p) for p in box_paths], ignore_index=True).to_csv(merged_boxes_csv, index=False)

feature_name = "B(mean+std, 1024)" if INCLUDE_STD else "A(mean only, 512)"
source_counts = source_table.groupby("source").size().rename("videos").to_frame()
print(f"model={clip_model}, frames_per_seg={frames_per_seg}, feature={feature_name}")
print("merged boxes:", merged_boxes_csv)
print(f"total videos: {len(source_table)}")
display(source_counts)
display(pd.concat([source_table.head(3), source_table.tail(3)], ignore_index=True))
display(source_table[source_table["source"] == "duck"])
assert len(source_table) == 37, f"Expected 37 videos (36+1), got {len(source_table)}"
assert source_counts.loc["wetlandbirds", "videos"] == 36
assert source_counts.loc["duck", "videos"] == 1


model=ViT-B/32, frames_per_seg=10, feature=B(mean+std, 1024)
merged boxes: C:\Users\osca0\Github\be-more-agent-duck\notebooks\outputs\cv_check_merged_boxes.csv
total videos: 37


,videos
source,
duck,1
wetlandbirds,36


,source,video_stem,video_path,label_path
0,wetlandbirds,027-gadwall,C:\Users\osca0\Github\be-more-agent-duck\data\...,C:\Users\osca0\Github\be-more-agent-duck\data\...
1,wetlandbirds,028-gadwall,C:\Users\osca0\Github\be-more-agent-duck\data\...,C:\Users\osca0\Github\be-more-agent-duck\data\...
2,wetlandbirds,029-gadwall,C:\Users\osca0\Github\be-more-agent-duck\data\...,C:\Users\osca0\Github\be-more-agent-duck\data\...
3,wetlandbirds,142-mallard,C:\Users\osca0\Github\be-more-agent-duck\data\...,C:\Users\osca0\Github\be-more-agent-duck\data\...
4,wetlandbirds,154-northern_shoveler,C:\Users\osca0\Github\be-more-agent-duck\data\...,C:\Users\osca0\Github\be-more-agent-duck\data\...
5,duck,manual,C:\Users\osca0\Github\be-more-agent-duck\data\...,C:\Users\osca0\Github\be-more-agent-duck\data\...


,source,video_stem,video_path,label_path
36,duck,manual,C:\Users\osca0\Github\be-more-agent-duck\data\...,C:\Users\osca0\Github\be-more-agent-duck\data\...


## 4. Extract crop features


In [27]:
X, y, groups = build_feature_matrix(
    video_paths,
    labels_csvs,
    clip_model,
    cache_path=None,
    frames_per_seg=frames_per_seg,
    include_std=INCLUDE_STD,
    boxes_csv=str(merged_boxes_csv),
    return_groups=True,
)
X = np.asarray(X)
y = np.asarray(y)
groups = np.asarray(groups)
source_names = np.array([source_by_group[g] for g in groups])
print("X:", X.shape, "y:", y.shape, "groups:", groups.shape)
print("class counts:")
print(pd.Series(y).value_counts().sort_index())
print("source counts:")
print(pd.Series(source_names).value_counts().sort_index())
print("source x label:")
display(pd.crosstab(pd.Series(source_names, name="source"), pd.Series(y, name="label")))
expected_dim = 1024 if INCLUDE_STD else 512
assert X.shape[1] == expected_dim
assert len(X) == len(y) == len(groups) == len(source_names)
assert "manual" in set(groups)
assert "duck" in set(source_names)


[16:09:39] [classifier] ── 영상 1/37: C:\Users\osca0\Github\be-more-agent-duck\data\train\wetlandbirds\videos\027-gadwall.mp4
[16:09:39] [classifier] crop 학습 영상 로드: C:\Users\osca0\Github\be-more-agent-duck\data\train\wetlandbirds\videos\027-gadwall.mp4
[16:09:39]   video_name=027-gadwall, labels=2개
[16:09:43] [classifier] crop 진행: 0/2 구간
[16:09:46] [classifier] ── 영상 2/37: C:\Users\osca0\Github\be-more-agent-duck\data\train\wetlandbirds\videos\028-gadwall.mp4
[16:09:46] [classifier] crop 학습 영상 로드: C:\Users\osca0\Github\be-more-agent-duck\data\train\wetlandbirds\videos\028-gadwall.mp4
[16:09:46]   video_name=028-gadwall, labels=3개
[16:09:52] [classifier] crop 진행: 0/3 구간
[16:10:05] [classifier] ── 영상 3/37: C:\Users\osca0\Github\be-more-agent-duck\data\train\wetlandbirds\videos\029-gadwall.mp4
[16:10:05] [classifier] crop 학습 영상 로드: C:\Users\osca0\Github\be-more-agent-duck\data\train\wetlandbirds\videos\029-gadwall.mp4
[16:10:05]   video_name=029-gadwall, labels=3개
[16:10:09] [classifier] cr

label,exploring,feeding,resting
source,,,
duck,12,11,12
wetlandbirds,67,44,38


## 5. StratifiedGroupKFold CV


In [28]:
le = LabelEncoder()
y_enc = le.fit_transform(y)
labels_order = sorted(np.unique(y))
n_groups = len(set(groups))
min_class_count = int(pd.Series(y).value_counts().min())
n_splits = min(FOLDS, n_groups, min_class_count)
if n_splits < 2:
    raise ValueError("Need at least two folds.")
sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)

macro_f1s = []
per_class = defaultdict(list)
manual_scores = []
empty_class_folds = defaultdict(int)
print(f"=== {n_splits}-Fold CV (group=video, {len(X)} segments / {n_groups} videos, feature={feature_name}) ===")
for fold, (tr, te) in enumerate(sgkf.split(X, y_enc, groups), 1):
    clf = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
    clf.fit(X[tr], y_enc[tr])
    pred_enc = clf.predict(X[te])
    y_true = y[te]
    y_pred = le.inverse_transform(pred_enc)
    mf1 = f1_score(y_true, y_pred, labels=labels_order, average="macro", zero_division=0)
    macro_f1s.append(mf1)
    for cls_name, cls_f1 in zip(labels_order, f1_score(y_true, y_pred, labels=labels_order, average=None, zero_division=0)):
        per_class[cls_name].append(float(cls_f1))
    for cls_name in labels_order:
        if cls_name not in set(y_true):
            empty_class_folds[cls_name] += 1
    te_groups = groups[te]
    te_sources = source_names[te]
    has_manual = bool(np.any(te_groups == "manual"))
    marker = " <-- manual test fold" if has_manual else ""
    print()
    print(f"fold {fold}: macro F1={mf1:.3f}{marker}")
    print(f"  groups={dict(pd.Series(te_groups).value_counts().sort_index())}")
    print(f"  sources={dict(pd.Series(te_sources).value_counts().sort_index())}")
    print(f"  label_dist={ {c: int((y_true == c).sum()) for c in labels_order} }")
    if has_manual:
        manual_mask = te_groups == "manual"
        manual_true = y_true[manual_mask]
        manual_pred = y_pred[manual_mask]
        manual_macro = f1_score(manual_true, manual_pred, labels=labels_order, average="macro", zero_division=0)
        manual_acc = accuracy_score(manual_true, manual_pred)
        manual_scores.append((manual_macro, manual_acc, int(manual_mask.sum()), fold))
        print("  manual-only report:")
        print(classification_report(manual_true, manual_pred, labels=labels_order, zero_division=0))
        print(f"  manual_only_macro_f1={manual_macro:.3f}, manual_only_accuracy={manual_acc:.3f}, n={manual_mask.sum()}")
print()
print(f"macro F1 = {np.mean(macro_f1s):.3f} +/- {np.std(macro_f1s):.3f}")
for cls_name in labels_order:
    warn = f"  [missing in {empty_class_folds[cls_name]} test fold(s)]" if empty_class_folds[cls_name] else ""
    print(f"  {cls_name:10} F1 = {np.mean(per_class[cls_name]):.3f} +/- {np.std(per_class[cls_name]):.3f}{warn}")
if manual_scores:
    manual_macro = np.array([score[0] for score in manual_scores])
    manual_acc = np.array([score[1] for score in manual_scores])
    manual_n = sum(score[2] for score in manual_scores)
    manual_folds = [score[3] for score in manual_scores]
    print(f"manual-test duck-only: folds={manual_folds}, macro_f1={manual_macro.mean():.3f} +/- {manual_macro.std():.3f}, accuracy={manual_acc.mean():.3f} +/- {manual_acc.std():.3f}, n={manual_n}")
else:
    print("manual-test duck-only: no fold had manual in test")


=== 5-Fold CV (group=video, 184 segments / 37 videos, feature=B(mean+std, 1024)) ===

fold 1: macro F1=0.594 <-- manual test fold
  groups={'029-gadwall': np.int64(3), '033-gadwall': np.int64(5), '034-gadwall': np.int64(7), '061-northern_shoveler': np.int64(4), '069-northern_shoveler': np.int64(3), '140-mallard': np.int64(2), '142-mallard': np.int64(4), 'manual': np.int64(35)}
  sources={'duck': np.int64(35), 'wetlandbirds': np.int64(28)}
  label_dist={np.str_('exploring'): 25, np.str_('feeding'): 19, np.str_('resting'): 19}
  manual-only report:
              precision    recall  f1-score   support

   exploring       0.44      0.67      0.53        12
     feeding       1.00      0.36      0.53        11
     resting       0.54      0.58      0.56        12

    accuracy                           0.54        35
   macro avg       0.66      0.54      0.54        35
weighted avg       0.65      0.54      0.54        35

  manual_only_macro_f1=0.542, manual_only_accuracy=0.543, n=35

fo